In [1]:
import pandas as pd
import numpy as np
import glob
from sklearn.model_selection import train_test_split
import os

In [2]:
# Create output directories if they don't exist
os.makedirs('ID Mapping/train', exist_ok=True)
os.makedirs('ID Mapping/val', exist_ok=True)
os.makedirs('ID Mapping/test', exist_ok=True)

# Get all CSV files from full_data directory
files = glob.glob('full_data/*.csv')

print(f"Found {len(files)} files to process\n")

Found 3 files to process



In [4]:
for file in files:
    # Extract filename without path and extension
    filename = os.path.basename(file)
    
    # Read the CSV file
    df = pd.read_csv(file)
    
    print(f"Processing {filename}...")
    print(f"  Total samples: {len(df)}")
    
    # Extract labels for stratification
    labels = df['label'] if 'label' in df.columns else None
    
    # First split: 70% train, 30% temp (which will be split into val and test)
    # 30% of 30% = 10% (val), 20% of 30% = 20% (test)
    train_idx, temp_idx = train_test_split(
        df.index, 
        test_size=0.30, 
        random_state=42,
        stratify=labels
    )
    
    # Second split: Split the 30% into 10% val and 20% test
    # 10/(10+20) = 0.333... for val, 20/(10+20) = 0.666... for test
    val_idx, test_idx = train_test_split(
        temp_idx, 
        test_size=0.667, 
        random_state=42,
        stratify=labels[temp_idx] if labels is not None else None
    )
    
    # Save train IDs (using dataframe index)
    train_ids = pd.Series(train_idx).reset_index(drop=True)
    train_ids.to_csv(f'ID Mapping/train/{filename}', index=False, header=['id'])
    
    # Save validation IDs (using dataframe index)
    val_ids = pd.Series(val_idx).reset_index(drop=True)
    val_ids.to_csv(f'ID Mapping/val/{filename}', index=False, header=['id'])
    
    # Save test IDs (using dataframe index)
    test_ids = pd.Series(test_idx).reset_index(drop=True)
    test_ids.to_csv(f'ID Mapping/test/{filename}', index=False, header=['id'])
    
    print(f"  Train: {len(train_idx)} samples ({len(train_idx)/len(df)*100:.1f}%)")
    print(f"  Val:   {len(val_idx)} samples ({len(val_idx)/len(df)*100:.1f}%)")
    print(f"  Test:  {len(test_idx)} samples ({len(test_idx)/len(df)*100:.1f}%)\n")

print("All splits completed successfully!")


Processing Arabic_1b_full.csv...
  Total samples: 3353
  Train: 2347 samples (70.0%)
  Val:   334 samples (10.0%)
  Test:  672 samples (20.0%)

Processing English_2e_full.csv...
  Total samples: 5647
  Train: 3952 samples (70.0%)
  Val:   564 samples (10.0%)
  Test:  1131 samples (20.0%)

Processing French_9a_full.csv...
  Total samples: 4014
  Train: 2809 samples (70.0%)
  Val:   401 samples (10.0%)
  Test:  804 samples (20.0%)

All splits completed successfully!
